# 《PythAPCS123》單元 13-5：語意錯誤（Logic Error）與常見邏輯盲點排查（WA 防範）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-5_logic_errors_and_wa_pitfalls.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：徹底克服競技程式中最讓人沮喪的「無聲殺手」——語意錯誤（Logic Error / Semantic Error）！當程式碼文法正確、執行過程沒有任何拋出崩潰（RE），但上傳至評判系統卻頻繁拿下刺眼的 Wrong Answer（WA）時，問題往往出在隱藏的邏輯死角。本單元深入剖析五大高頻邏輯暗坑：差一錯誤（Off-by-one）、運算子優先級失控、二進位浮點數精度截斷、二維串列淺拷貝幽靈修改、以及變數命名遮蔽內建函式。學會精準診斷思維盲區，全面築起抵禦 WA 的鋼鐵防線。


### 13.5.1 什麼是語意錯誤？語法合法但運算邏輯背離題意的本質

在程式設計的世界中，如果說語法錯誤（SyntaxError）就像是「拼錯英文單字或漏打標點」，執行錯誤（Runtime Error）像是「開車開到懸崖邊翻車」，那麼**語意錯誤（Logic Error / Semantic Error）**就如同「你非常流暢地開著一輛性能完美的車，但導航目的地完全設定錯了方向」。

語意錯誤之所以被稱為競賽中最棘手的「隱形殺手」，原因在於它具有高度的隱蔽性：
1. **編譯器與直譯器完全沉默**：因為所有的單字、縮排、括號與型態完全符合 Python 規範，直譯器絕對不會主動跳出任何一行紅字警告。
2. **本機執行看似順利**：你隨手測了一筆最簡單的樣例，可能「剛好」碰巧算出看似正常的數字，但一旦遇到多樣化的隱藏測試點，程式就會原形畢露，給出完全錯誤的答案，被裁判系統無情判定為 **WA（Wrong Answer）**。

電腦是極其忠實的僕人，它永遠只會嚴格執行你「寫給它的指令」，而不是你「腦袋裡以為你寫的指令」。要消滅語意錯誤，不能依賴直譯器報錯，而是必須具備「追蹤變數狀態」與「嚴謹邏輯推導」的能力。


In [ ]:
# 13.5.1 程式碼演示：語法合法但邏輯背離題意的經典對比
# 題目需求：計算串列中所有「大於 10 且小於 20」之數值的總和

data = [5, 12, 18, 20, 25]

# 錯誤示範（語法完全正確，但條件寫錯成 or，邏輯徹底偏離題意）
def flawed_sum(nums):
    total = 0
    for x in nums:
        # 邏輯致命傷：原意是大於 10「且」小於 20，卻手滑寫成 or
        if x > 10 or x < 20:
            total += x
    return total

# 正確寫法
def correct_sum(nums):
    total = 0
    for x in nums:
        if 10 < x < 20: # 嚴謹鏈式比較：大於 10 且小於 20
            total += x
    return total

print("原始陣列:", data)
print("錯誤函式計算結果:", flawed_sum(data)) # 算出來是全部數字相加 = 80 (WA!)
print("正確函式計算結果:", correct_sum(data)) # 僅 12 + 18 = 30 (AC!)
print("結論：直譯器不會對錯誤的邏輯提出抗議，除錯責任完全在程式設計師身上！")


### 13.5.1 語法重點回顧與核心觀念提煉

語意錯誤的診斷與防護心法：
1. **拋棄「沒報錯就代表寫對」的幻覺**：沒有錯誤訊息僅代表 Python 能理解你的語法，絕不代表演算法邏輯正確。
2. **手動追蹤小規模資料（Trace Table）**：在遇到結果不合預期時，取 3 到 5 個數字，拿張紙筆逐行手動模擬變數變化，是抓出邏輯偏差最有效的手段。
3. **題意逐字覆核**：特別留意題目中的「大於（`>`）」vs「大於等於（`>=`）」、「包含」vs「不包含」、「且」vs「或」。


In [ ]:
# 13.5.1 學生實作練習：及格且未滿百之分數篩選器
# 任務說明：實作 filter_passing_scores(scores) 函式
# 題意要求：從整數串列 scores 中篩選出「大於等於 60 分且小於 100 分」的成績，回傳新串列
# 請小心排查邏輯條件，絕對不能將滿分 100 納入，也不可遺漏 60 分的邊界值！

def filter_passing_scores(scores: list) -> list:
    res = []
    # 請在此處撰寫正確的邏輯判斷
    for s in scores:
        if 60 <= s < 100:
            res.append(s)
    return res

# 測試用例
test_scores = [59, 60, 75, 99, 100, 105]
print("篩選結果:", filter_passing_scores(test_scores))


In [ ]:
# 13.5.1 單元測試驗證
assert filter_passing_scores([59, 60, 75, 99, 100]) == [60, 75, 99]
assert filter_passing_scores([100, 100, 50]) == []
assert filter_passing_scores([60]) == [60]
assert filter_passing_scores([]) == []
print("13.5.1 單元測試全數通過！")


### 13.5.2 邏輯盲點一：差一錯誤（Off-by-one Error）

在電腦科學中，有一句著名的黑色幽默名言：「電腦科學領域只有兩大難題：快取失效、命名，以及**差一錯誤（Off-by-one Error, OBOE）**。」

差一錯誤指的是：你的演算法架構完全正確，但在迴圈計數、範圍設定、開閉區間或索引對齊時，剛好「多算了一次」或「少算了一次」。在 Python 中，差一錯誤最常潛伏在以下兩個角落：
1. **`range(start, stop)` 的「左閉右開」特性**：Python 的 `range(a, b)` 永遠「包含起點 $a$，但不包含終點 $b$」（數學符號 $[a, b)$）。初學者若想從 $1$ 跑到 $10$，直覺寫下 `range(1, 10)`，迴圈只會跑到 $9$ 就收工，導致最後一筆資料被無辜丟失（少算一次）。
2. **陣列長度與下標轉換**：長度為 $N$ 的陣列最後一個下標是 $N-1$；當需要遍歷全體元素時，`range(len(arr))` 是正確的，但若寫成 `range(1, len(arr))` 則漏了開頭；若寫成 `range(len(arr) - 1)` 則漏了結尾。

克服差一錯誤的法寶就是：**在下筆時永遠在心裡代入「第一個值」與「最後一個值」做極限驗算**。


In [ ]:
# 13.5.2 程式碼演示：range 左閉右開之差一陷阱
print("--- 需求：計算 1 加到 10 的總和（正確答案為 55）---")

# 差一錯誤示範：range(1, 10) 只會跑到 9
faulty_sum = sum(range(1, 10))
print(f"誤寫為 range(1, 10) 的結果: {faulty_sum} ❌ (少了最後一個數字 10！)")

# 正確寫法：終止值必須寫 11
correct_total = sum(range(1, 11))
print(f"修正為 range(1, 11) 的結果: {correct_total} ✅")

print("\n--- 字串切片中的差一陷阱 ---")
text = "APCS_PYTHON"
print(f"原始字串: {text}")
# 需求：取出前 4 個字元 'APCS'
# 錯誤思維：前 4 個字元，索引從 0 到 3，若寫 text[0:3] 只會取到索引 2
print(f"誤寫 text[0:3]: '{text[0:3]}' (少取 1 個字元)")
print(f"正確 text[0:4]: '{text[0:4]}' (正確涵蓋索引 0, 1, 2, 3)")


### 13.5.2 語法重點回顧與核心觀念提煉

差一錯誤的四步防禦口訣：
1. **起點檢驗**：迴圈的第 1 次迭代，變數的值是多少？是否吻合題目的起始條件？
2. **終點檢驗**：迴圈的最後 1 次迭代，變數的值是多少？是否精準抵達期望的終點邊界？
3. **元素個數檢驗**：對於區間 $[a, b]$，包含端點的整數個數為 $b - a + 1$；若使用 `range(a, b + 1)`，其產生的元素個數剛好也是 $(b + 1) - a = b - a + 1$。
4. **切片長度檢驗**：對於 `s[start:end]`，切出來的字串長度必定恆等於 `end - start`。


In [ ]:
# 13.5.2 學生實作練習：精準包含端點之倍數統計器
# 任務說明：實作 count_multiples_in_range(start, end, k) 函式
# 計算在整數閉區間 [start, end] 內（包含 start 與 end 本身）
# 有多少個整數是 k 的倍數（x % k == 0）
# 請小心處理 range 的終點設定，絕不可因差一錯誤而漏掉 end 端點！

def count_multiples_in_range(start: int, end: int, k: int) -> int:
    count = 0
    # 請在此處使用正確的 range 上下界
    for num in range(start, end + 1):
        if num % k == 0:
            count += 1
    return count

# 測試用例
print("區間 [1, 10] 內 5 的倍數個數:", count_multiples_in_range(1, 10, 5))
print("區間 [3, 9] 內 3 的倍數個數:", count_multiples_in_range(3, 9, 3))


In [ ]:
# 13.5.2 單元測試驗證
assert count_multiples_in_range(1, 10, 5) == 2 # 5, 10
assert count_multiples_in_range(3, 9, 3) == 3   # 3, 6, 9
assert count_multiples_in_range(1, 4, 5) == 0
assert count_multiples_in_range(10, 10, 10) == 1
print("13.5.2 單元測試全數通過！")


### 13.5.3 邏輯盲點二：運算優先級陷阱

當你在同一行程式碼中混合使用了算術運算（`+`, `-`, `*`, `//`）、比較運算（`==`, `<`, `>=`）以及邏輯運算（`not`, `and`, `or`）時，直譯器會依照嚴格的**「運算子優先級（Operator Precedence）」**排定執行順序。如果忽視了優先級，程式算出來的結果將完全失控！

考場最高頻的三大優先級災難：
1. **加法與整數除法：`a + b // 2` 陷阱**：
   在計算兩數平均或二分搜尋中點時，初學者常寫下 `mid = a + b // 2`。因為除法 `//` 優先於加法 `+`，電腦實際執行的是 $a + \frac{b}{2}$，而非 $\frac{a + b}{2}$！正確寫法「必須加括號」：`mid = (a + b) // 2`。
2. **位元運算與比較運算**：
   若使用位元運算判斷奇偶數，`x & 1 == 1` 會因比較運算子 `==` 優先於位元運算子 `&`，導致電腦先執行 `1 == 1`（為 True），再計算 `x & True`，引爆邏輯錯誤！正確寫法為 `(x & 1) == 1`。
3. **邏輯 `not`、`and`、`or` 混用**：
   Python 的優先順序為：`not` 最優先，其次是 `and`，最後才是 `or`。因此 `A or B and C` 等價於 `A or (B and C)`。若未加括號，常導致條件分支走入完全相反的方向。

記住競賽防身第一金律：**「有任何疑慮，直接加上括號 `()`，括號是免費的，也是絕對清晰的保證！」**


In [ ]:
# 13.5.3 程式碼演示：優先級失控對比
# 案例 1: 平均值計算
left = 10
right = 20

wrong_mid = left + right // 2 # 相當於 10 + (20 // 2) = 20
correct_mid = (left + right) // 2 # (10 + 20) // 2 = 15

print(f"中點計算 [10, 20]:")
print(f"  漏括號 left + right // 2 = {wrong_mid} ❌ (WA)")
print(f"  加括號 (left + right) // 2 = {correct_mid} ✅ (AC)")

# 案例 2: 邏輯 and / or 結合順序
# 需求：會員必須滿 18 歲，且必須具備 VIP 或員工資格才可進入
is_adult = True
is_vip = False
is_staff = True

# 錯誤寫法：未加括號
# Python 解讀為: (is_adult and is_vip) or is_staff
# 只要是員工，就算不是大人也能進？若 is_adult 為 False 就會出大事！
res_ambiguous = is_adult and is_vip or is_staff

# 正確寫法：清楚用括號表達業務邏輯
res_explicit = is_adult and (is_vip or is_staff)

print(f"\n邏輯資格判定:")
print(f"  明確括號結果: {res_explicit} ✅")


### 13.5.3 語法重點回顧與核心觀念提煉

運算優先級記憶口訣與防身指南：
1. **括號小宇宙優先**：小括號 `()` 享有全語言最高優先級，能隨時打破任何預設順序。
2. **算術優先級四階層**：
   - 第一階：次方 `**`
   - 第二階：正負號 `+x`, `-x`
   - 第三階：乘除模 `*`, `/`, `//`, `%`
   - 第四階：加減 `+`, `-`
3. **比較優於邏輯**：`<`, `>`, `==` 先算，之後才算 `not` -> `and` -> `or`。
4. **競賽鐵律**：不要為了炫技而省略括號！多加一對括號既不影響執行效能，又能讓邏輯堅如磐石，更能大幅降低閱卷時的思維負擔。


In [ ]:
# 13.5.3 學生實作練習：安全折半搜尋中點與資格審核
# 任務說明：實作 calculate_midpoint_and_check(a, b, threshold) 函式
# 1. 計算兩非負整數 a 與 b 的平均值 mid（使用整數除法 // 取下高斯）
# 2. 判斷 mid 是否大於 threshold，或者 (a 和 b 兩者皆大於 threshold)
# 3. 回傳 (mid, is_qualified) 元組
# 務必加上正確的括號，杜絕一切優先級造成的 WA！

def calculate_midpoint_and_check(a: int, b: int, threshold: int) -> tuple:
    # 請在此處妥善使用括號計算
    mid = (a + b) // 2
    is_qualified = (mid > threshold) or ((a > threshold) and (b > threshold))
    return (mid, is_qualified)

# 測試用例
print("測試 (10, 20, 12):", calculate_midpoint_and_check(10, 20, 12))
print("測試 (4, 6, 5):", calculate_midpoint_and_check(4, 6, 5))


In [ ]:
# 13.5.3 單元測試驗證
assert calculate_midpoint_and_check(10, 20, 12) == (15, True)
assert calculate_midpoint_and_check(10, 20, 15) == (15, False)
assert calculate_midpoint_and_check(0, 100, 49) == (50, True)
assert calculate_midpoint_and_check(6, 8, 5) == (7, True)
print("13.5.3 單元測試全數通過！")


### 13.5.4 邏輯盲點三：浮點數精確度陷阱

在初學者的常識中，$0.1 + 0.2$ 必然等於 $0.3$。然而，當你在 Python 直譯器中輸入 `0.1 + 0.2 == 0.3` 時，它卻會回傳驚人的 **`False`**！

這不是 Python 的臭蟲，而是所有採用 IEEE 754 浮點數標準的現代電腦無法避免的物理宿命。因為電腦底層使用二進位儲存資料，就像十進位無法精確表示 $\frac{1}{3} = 0.33333...$ 一樣，二進位也無法精確表示十進位的 $0.1$ 與 $0.2$（它們在二進位中是無限循環小數）。在進行運算時，直譯器會截斷末位，導致 $0.1 + 0.2$ 的實際數值為 `0.30000000000000004`。

在 APCS 考場上，這會引爆兩大致命後果：
1. **浮點數直接以 `==` 比大小**：一旦牽涉到小數加減乘除，直接用 `a == b` 比對判斷幾乎必定得到 `False`，導致該進的分支進不去。
2. **整數問題誤用浮點除法**：題目若要求整數答案，寫了單斜線 `/`（會自動升級為浮點數 `float`），一旦數值非常龐大（例如大整數大於 $2^{53}$），浮點數的有效位數不足就會發生「精確度丟失（Precision Loss）」，直接把大數末幾位吃掉！

解法：
- 比較兩浮點數是否相等時，應判斷「兩者差值的絕對值是否小於極小值 $\epsilon$（如 $10^{-9}$）」：`abs(a - b) < 1e-9`。
- 能用整數算就絕不用浮點數算（例如將金額乘上 100 轉為「分」進行純整數運算）。


In [ ]:
# 13.5.4 程式碼演示：浮點數精度丟失與安全比較
import math

print("--- 經典浮點數精度偏差 ---")
val = 0.1 + 0.2
print(f"0.1 + 0.2 的真實記憶體數值: {repr(val)}")
print(f"直接使用 == 0.3 比對結果: {val == 0.3} ❌ (WA 殺手！)")

# 防禦方案一：手動容忍極小誤差 epsilon
EPSILON = 1e-9
is_equal_manual = abs(val - 0.3) < EPSILON
print(f"使用 abs(a - b) < 1e-9 比對結果: {is_equal_manual} ✅")

# 防禦方案二：使用標準函式庫 math.isclose()
is_equal_math = math.isclose(val, 0.3)
print(f"使用 math.isclose() 比對結果: {is_equal_math} ✅")

print("\n--- 大整數除法 vs 浮點除法精準度 ---")
big_num = 10**18 + 1
# 使用整數除法保持完全精確
int_div = big_num // 1
# 使用浮點除法可能丟失精度
float_div = int(big_num / 1.0)

print(f"原始大整數: {big_num}")
print(f"整數除法保留: {int_div}")
print(f"浮點除法後轉整數: {float_div}")
print(f"浮點除法是否失真: {big_num != float_div}")


### 13.5.4 語法重點回顧與核心觀念提煉

浮點數精準度防禦三大準則：
1. **嚴禁使用 `==` 比較浮點數**：競賽中比較兩浮點數是否相等，標準寫法一律是：
   ```python
   if abs(a - b) < 1e-7: # AC
   ```
2. **整數運算保持純潔性**：若題目涉及網格座標、倍數、計數、整數下標，一律使用整數除法 `//`，嚴禁使用單斜線 `/` 後再轉 `int()`，以避免浮點截斷偏差。
3. **乘除化整技巧**：例如比較 $\frac{a}{b}$ 與 $\frac{c}{d}$ 的大小，若 $b, d > 0$，請在紙上化簡為交叉相乘：比較 $a \times d$ 與 $c \times b$，將除法全數轉換為無損精度的純整數乘法！


In [ ]:
# 13.5.4 學生實作練習：安全浮點相等判定器
# 任務說明：實作 is_float_equal(a, b, eps=1e-9) 函式
# 判斷兩浮點數 a 與 b 在指定容許誤差 eps 範圍內是否視為相等
# 若差值小於 eps 回傳 True，否則回傳 False

def is_float_equal(a: float, b: float, eps: float = 1e-9) -> bool:
    # 請在此處實作 epsilon 容差比對
    return abs(a - b) < eps

# 測試用例
print("0.1 + 0.2 是否等於 0.3:", is_float_equal(0.1 + 0.2, 0.3))
print("0.5 是否等於 0.5000000001:", is_float_equal(0.5, 0.5000000001))


In [ ]:
# 13.5.4 單元測試驗證
assert is_float_equal(0.1 + 0.2, 0.3) == True
assert is_float_equal(1.0, 1.0) == True
assert is_float_equal(1.0, 1.05) == False
assert is_float_equal(1e-10, 0.0) == True
print("13.5.4 單元測試全數通過！")


### 13.5.5 邏輯盲點四：淺拷貝共享記憶體幽靈

在 APCS 實作題第二題中，二維陣列（地圖、棋盤、網格）是極具代表性的基礎資料結構。許多考生在初始化一個 $3 \times 3$ 的全零矩陣時，為了圖方便，常寫出以下這句看似精練的代碼：
```python
# 致命幽靈寫法：千萬不要這樣建立二維串列！
grid = [[0] * 3] * 3
```

當你嘗試執行 `grid[0][0] = 9`，滿心以為只有左上角第一格會變成 9 時，印出整個矩陣卻會讓你目瞪口呆：**第一列、第二列、第三列的第一格全都同時變成了 9！**

這就是讓無數考生痛失分數的**「淺拷貝共享記憶體幽靈（Shared Reference Trap）」**。
原因在於：`[0] * 3` 建立了一個長度為 3 的一維串列物件；而外層的 `* 3` 並沒有建立三個「獨立的串列」，而是把同一個一維串列的「記憶體參照（Reference）」複製了三份！換言之，`grid[0]`、`grid[1]`、`grid[2]` 底層指涉的是同一塊實體記憶體，牽一髮而動全身。

同理，若寫 `b = a`，`b` 只是 `a` 的別名，修改 `b[0]` 會直接竄改 `a[0]`。
正確建立二維網格的標準寫法必須使用**串列生成式（List Comprehension）**：
```python
grid = [[0] * 3 for _ in range(3)] # 每一列都是獨立全新產生的串列物件！
```


In [ ]:
# 13.5.5 程式碼演示：幽靈連動修改 vs 獨立網格生成
print("--- 錯誤寫法：[[0] * 3] * 3 記憶體共享慘劇 ---")
bad_grid = [[0] * 3] * 3
print("修改前 bad_grid:", bad_grid)
# 嘗試只修改第 0 列、第 0 欄
bad_grid[0][0] = 9
print("修改 bad_grid[0][0] = 9 後:")
for row in bad_grid:
    print(" ", row)
print("驚悚發現：每一列的第一格通通變成 9，因為三列的 id 相同！")
print(f"id(row0)={id(bad_grid[0])}, id(row1)={id(bad_grid[1])}")

print("\n--- 正確寫法：使用生成式生成獨立各列 ---")
good_grid = [[0] * 3 for _ in range(3)]
good_grid[0][0] = 9
print("修改 good_grid[0][0] = 9 後:")
for row in good_grid:
    print(" ", row)
print(f"id(row0)={id(good_grid[0])}, id(row1)={id(good_grid[1])} (記憶體位址各不相同！)")


### 13.5.5 語法重點回顧與核心觀念提煉

可變物件參照防禦兩大原則：
1. **二維網格嚴禁乘號巢狀複製**：內層複製值 `[val] * cols` 是安全的（因為整數是不可變型別），但外層複製列必須強制使用生成式：
   ```python
   # H 列、W 欄的正確安全初始化模板：
   grid = [[0] * W for _ in range(H)]
   ```
2. **複製串列切斷血緣**：
   - 若要複製一維串列，使用 `b = a.copy()` 或 `b = a[:]`。
   - 若要完全獨立複製包含巢狀結構的多維串列，使用 `import copy; b = copy.deepcopy(a)`。


In [ ]:
# 13.5.5 學生實作練習：安全建立與更新二維網格
# 任務說明：實作 create_and_mark_grid(H, W, r, c, val) 函式
# 1. 建立一個高為 H、寬為 W，初值全為 0 的獨立二維網格
# 2. 將座標 (r, c) 的數值更新為 val
# 3. 回傳該網格
# 嚴格驗證：除了 (r, c) 外，其餘位置必須維持 0，絕不可連動修改！

def create_and_mark_grid(H: int, W: int, r: int, c: int, val: int) -> list:
    # 請使用正確的串列生成式建立網格，杜絕幽靈連動
    grid = [[0] * W for _ in range(H)]
    grid[r][c] = val
    return grid

# 測試用例
my_grid = create_and_mark_grid(3, 4, 1, 2, 7)
print("建立並標記網格結果:")
for row in my_grid:
    print(" ", row)


In [ ]:
# 13.5.5 單元測試驗證
g = create_and_mark_grid(3, 3, 0, 0, 9)
assert g[0][0] == 9
assert g[1][0] == 0, "檢驗第 1 列未受連動影響"
assert g[2][0] == 0, "檢驗第 2 列未受連動影響"
assert sum(sum(row) for row in g) == 9, "整個網格應僅有唯一的 9"
print("13.5.5 單元測試全數通過！")


### 13.5.6 邏輯盲點五：變數遮蔽與污染（同名覆蓋內建函數）

初學者在解題時，常常根據最直觀的英文來替變數命名。例如想計算總和，就隨手宣告 `sum = 0`；想存一個清單，就宣告 `list = [1, 2, 3]`；想找最大值，就宣告 `max = -1`。

在 Python 中，這種命名方式隱藏著巨大的後續災難——**「內建識別字遮蔽（Shadowing Built-in Functions）」**！
Python 的設計極度自由，內建函數（如 `sum`, `list`, `max`, `min`, `str`, `int`）本質上只是全域命名空間中的一個函式物件。當你寫下 `sum = 0` 時，你等於是親手「用整數 0 覆蓋掉了 Python 內建的 `sum` 函式」！
在當下的這一行不會報錯，但當你之後在程式碼第 20 行想呼叫 `total = sum(my_list)` 求和時，直譯器會當場引爆令人匪夷所思的：
```
TypeError: 'int' object is not callable
```
這常讓初學者抓狂：「我明明呼叫的是 Python 內建的 sum，怎麼會說整數不能被呼叫？」

養成良好的命名自律：想存總和請命名為 `total` 或 `total_sum`；想存串列請命名為 `arr`, `nums`, `items`；想存極值請命名為 `max_val`, `min_val`，絕不侵犯 Python 內建的神聖關鍵字！


In [ ]:
# 13.5.6 程式碼演示：遮蔽內建函式的連鎖悲劇
print("--- 壞習慣示範：使用 sum 作為變數名稱 ---")
# 正常呼叫內建 sum
print("正常內建 sum([10, 20]):", sum([10, 20]))

# 覆蓋內建名稱：給予整數賦值
sum = 100
print(f"宣告了自訂變數 sum = {sum}")

# 嘗試再次呼叫 sum() 求總和
try:
    bad_call = sum([1, 2, 3])
except TypeError as e:
    print(f"[災難現場] 拋出 TypeError: {e}")
    print("說明：因為內建的 sum 函式已被你覆蓋成整數 100，整數無法當作函式呼叫！")

# 修復被污染的命名空間（刪除自訂變數，還原內建函式）
del sum
print(f"刪除自訂變數後，還原內建 sum([1, 2, 3]) = {sum([1, 2, 3])} ✅")


### 13.5.6 語法重點回顧與核心觀念提煉

考場安全命名清單與避坑指南：
1. **絕對禁用的變數名稱列表**：
   - 數值相關：`sum`, `max`, `min`, `abs`, `round`, `pow`
   - 型態相關：`list`, `dict`, `set`, `str`, `int`, `float`, `tuple`
   - 走訪相關：`iter`, `range`, `len`, `enumerate`, `zip`, `map`, `filter`, `input`
2. **推薦的替換命名慣用詞**：
   - 總和：`total`, `ans`, `sum_val`
   - 串列：`arr`, `nums`, `elements`, `records`
   - 極值：`max_val`, `best_score`, `min_cost`
   - 字串：`s`, `text`, `line`
3. **IDE 與 Colab 視覺提示**：當你在輸入變數名稱時，如果發現編輯器將它標示為特定的保留字顏色（如紫色或綠色），代表它很可能是內建關鍵字，請立刻在名稱後方加上 `_val` 或換個單字！


In [ ]:
# 13.5.6 學生實作練習：安全極值與總和統計器
# 任務說明：實作 calculate_stats_safely(nums) 函式
# 給定非空整數串列 nums，回傳字典包含 "sum", "max", "min", "avg" 四個鍵
# 嚴格要求：函式內部的變數命名絕對不可使用 sum, max, min, list 等內建名稱！
# 必須正確呼叫內建函式完成計算！

def calculate_stats_safely(nums: list) -> dict:
    # 請在此處使用安全的變數名稱進行計算
    total_val = sum(nums)
    max_val = max(nums)
    min_val = min(nums)
    avg_val = total_val / len(nums)
    
    return {
        "sum": total_val,
        "max": max_val,
        "min": min_val,
        "avg": avg_val
    }

# 測試用例
sample_data = [10, 20, 30, 40]
print("統計結果:", calculate_stats_safely(sample_data))


In [ ]:
# 13.5.6 單元測試驗證
res = calculate_stats_safely([1, 2, 3, 4, 5])
assert res["sum"] == 15
assert res["max"] == 5
assert res["min"] == 1
assert res["avg"] == 3.0
print("13.5.6 單元測試全數通過！")


## 13.5 總結與 WA 防禦地圖

在本單元中，我們地毯式排查了無聲無息但殺傷力極大的五大語意錯誤（Logic Error）。當你上傳程式碼得到 WA 時，請按照以下地圖逐項自我檢查：

| 邏輯盲點類別 | 典型出錯特徵 | 考場防禦黃金法則 |
| :--- | :--- | :--- |
| **差一錯誤（Off-by-one）** | `range` 漏掉最後一筆、切片長度不符 | 帶入極限值驗算：開閉區間 $[a, b]$ 元素個數為 $b - a + 1$ |
| **運算優先級混亂** | `a + b // 2` 算錯中點、邏輯 `and/or` 偏差 | **括號是免費的！** 有疑慮一律明確加上 `( )` |
| **浮點數精度丟失** | `0.1 + 0.2 == 0.3` 判定失敗、大整數失真 | 嚴禁 `==` 比浮點數，改用 `abs(a - b) < 1e-9`；能用整數除 `//` 就不用 `/` |
| **淺拷貝共享幽靈** | `[[0]*W]*H` 修改一格全網格連鎖變動 | 建立多維容器嚴禁外層乘號，強制使用列表生成式 `[... for _ in range(H)]` |
| **變數遮蔽污染** | 宣告 `sum = 0` 導致後續 `sum()` 噴出 TypeError | 絕對自律命名：改用 `total`, `max_val`, `arr`，避開內建函數名 |

### 🚀 下一步學習指引
消滅了語法錯誤、崩潰例外與答案錯誤之後，在 APCS 考場上還有最後一座難以逾越的高山：**「時間超限（Time Limit Exceeded, TLE）」**！
明明演算法能算對答案，但測資一擴大到 $N=10^5$，程式碼執行超過 1 秒就被評判伺服器強制切斷。
在下一單元 **13-6《時間超限（TLE）診斷：運算量估算、無窮迴圈與隱形效能坑洞防制》** 中，我們將建立每秒 $10^7$ 次操作的直覺模型，徹底剷除 `list in` 與字串串接等隱形效能黑洞！
